In [ ]:
# ==============================================================================
# CELL 1: Environment Setup & Library Installation
# ==============================================================================
# Install the core unsloth package
#!pip install unsloth
#!pip install unsloth_zoo

# Install dependencies with strict version caps to satisfy Unsloth's requirements
#!pip install peft accelerate bitsandbytes "trl<=1.13.0" "datasets<5.0.0" scikit-learn "pandas==2.2.3" "numpy<2.3" tqdm xformers

In [ ]:
# ==============================================================================
# CELL 2: Data Loading, Splitting & Explicit Verification
# ==============================================================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Load dataset directly from the official GitHub raw repository
github_raw_url = "https://raw.githubusercontent.com/LAPHR/Travel-Dataset-5000-/master/5000TravelQuestionsDataset.csv"

# Load without header since raw CSV contains no column titles
df = pd.read_csv(
    github_raw_url,
    encoding='latin1',
    header=None,
    names=['Question', 'Coarse_Class', 'Fine_Class']
)

print(f"Total dataset loaded: {len(df)} samples")

# 2. Data Cleaning: Drop the fine-grain class column
df = df.drop(columns=['Fine_Class'])
df['Question'] = df['Question'].astype(str).str.strip()
df['Coarse_Class'] = df['Coarse_Class'].astype(str).str.strip()

print(f"Features retained: {df.columns.tolist()}")

# 3. Data Splitting: 4000 train, 300 validation, 700 test
# First isolate 700 test samples
train_val_df, test_df = train_test_split(
    df,
    test_size=700,
    random_state=42,
    stratify=df['Coarse_Class']
)

# Next split remaining 4300 samples into 4000 train and 300 validation
train_df, val_df = train_test_split(
    train_val_df,
    test_size=300,
    random_state=42,
    stratify=train_val_df['Coarse_Class']
)

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# 4. Explicit Split Size & Stratification Verification
print("\n--- 1. Data Splits Established ---")
print(f"Training samples   : {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Validation samples : {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"Testing samples    : {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

print("\n--- 2. Class Distribution Across Splits (Stratification Proof) ---")
split_verification_df = pd.DataFrame({
    'Train Count': train_df['Coarse_Class'].value_counts(),
    'Train %': (train_df['Coarse_Class'].value_counts(normalize=True) * 100).round(2),
    'Val Count': val_df['Coarse_Class'].value_counts(),
    'Val %': (val_df['Coarse_Class'].value_counts(normalize=True) * 100).round(2),
    'Test Count': test_df['Coarse_Class'].value_counts(),
    'Test %': (test_df['Coarse_Class'].value_counts(normalize=True) * 100).round(2),
})
print(split_verification_df.to_markdown())

print("\n--- 3. Data Leakage / Overlap Check ---")
train_questions = set(train_df['Question'])
test_questions = set(test_df['Question'])
val_questions = set(val_df['Question'])

train_test_overlap = len(train_questions.intersection(test_questions))
train_val_overlap = len(train_questions.intersection(val_questions))
print(f"Overlapping questions between Train and Test: {train_test_overlap}")
print(f"Overlapping questions between Train and Val:  {train_val_overlap}")

print("\n--- 4. Cleaned Sample Inspection (First 3 Train Rows) ---")
print(train_df.head(3).to_markdown(index=False))

Total dataset loaded: 5000 samples
Features retained: ['Question', 'Coarse_Class']

--- 1. Data Splits Established ---
Training samples   : 4000 (80.0%)
Validation samples : 300 (6.0%)
Testing samples    : 700 (14.0%)

--- 2. Class Distribution Across Splits (Stratification Proof) ---
| Coarse_Class   |   Train Count |   Train % |   Val Count |   Val % |   Test Count |   Test % |
|:---------------|--------------:|----------:|------------:|--------:|-------------:|---------:|
| TGU            |           976 |     24.4  |          73 |   24.33 |          171 |    24.43 |
| TTD            |           911 |     22.78 |          69 |   23    |          160 |    22.86 |
| TRS            |           809 |     20.23 |          61 |   20.33 |          141 |    20.14 |
| ACM            |           576 |     14.4  |          43 |   14.33 |          101 |    14.43 |
| FOD            |           417 |     10.42 |          31 |   10.33 |           73 |    10.43 |
| ENT            |           173 | 

In [ ]:
# ==============================================================================
# CELL 3: Load 4-bit Quantized LLM & Tokenizer via Unsloth
# ==============================================================================
import torch
from unsloth import FastLanguageModel

# Model configuration
max_seq_length = 1024
dtype = None # Auto-detection (bfloat16 for modern Ampere+, float16 for T4)
load_in_4bit = True # 4-bit quantization to fit easily within 16GB T4 VRAM

# Load Llama-3-8B-Instruct (pre-quantized 4-bit)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Enable fast inference mode
FastLanguageModel.for_inference(model)
print("Model and Tokenizer successfully loaded in 4-bit precision.")

ModuleNotFoundError: No module named 'unsloth'

In [ ]:
# ==============================================================================
# CELL 4: Dynamic Prompt Formatting (0-shot, 1-shot, 3-shot)
# ==============================================================================
SYSTEM_PROMPT = (
    "You are a precise text classification system for the travel domain. "
    "Classify the user's travel question into exactly one of the following coarse-grain categories:\n"
    "- TTD: things to do\n"
    "- TGU: travel guide\n"
    "- ACM: accommodation\n"
    "- TRS: transport\n"
    "- WTH: weather\n"
    "- FOD: food\n"
    "- ENT: entertainment\n\n"
    "Output ONLY the 3-letter category abbreviation. Do not output any explanations or extra words."
)

VALID_CLASSES = ['TTD', 'TGU', 'ACM', 'TRS', 'WTH', 'FOD', 'ENT']

def build_prompt(question, shot_examples=[], system_prompt=SYSTEM_PROMPT):
    """
    Constructs a Llama-3 chat template dynamically incorporating few-shot exemplars.
    """
    prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|>"

    # Inject few-shot training examples if provided
    for ex in shot_examples:
        prompt += (
            f"<|start_header_id|>user<|end_header_id|>\n\nQuestion: {ex['Question']}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n{ex['Coarse_Class']}<|eot_id|>"
        )

    # Target query
    prompt += (
        f"<|start_header_id|>user<|end_header_id|>\n\nQuestion: {question}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    return prompt

In [ ]:
# ==============================================================================
# CELL 5: Batched Inference Engine & F1 Evaluation Helper
# ==============================================================================
import torch
import re
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
from unsloth import FastLanguageModel

tokenizer.padding_side = 'left'

def extract_label(generated_text):
    clean_text = generated_text.strip().upper()
    for cls in VALID_CLASSES:
        if re.search(r'\b' + cls + r'\b', clean_text):
            return cls
    for cls in VALID_CLASSES:
        if cls in clean_text:
            return cls
    return "UNKNOWN"

def verify_token_limits(prompts, max_len=8192):
    tokens = tokenizer(prompts, return_tensors="pt", padding=True)
    max_batch_length = tokens['input_ids'].shape[1]
    if max_batch_length > max_len:
        raise ValueError(f"CRITICAL: Prompt length ({max_batch_length}) exceeds model limit!")
    print(f"Token check passed. Max sequence length: {max_batch_length} tokens.")

def run_evaluation(eval_df, shot_examples=[], batch_name="Evaluation", batch_size=8, system_prompt=SYSTEM_PROMPT):
    FastLanguageModel.for_inference(model)
    predictions = []
    ground_truth = eval_df['Coarse_Class'].tolist()

    print(f"\nRunning {batch_name} on {len(eval_df)} samples (Batched)...")

    # Build prompts using the specified system prompt
    prompts = [build_prompt(row['Question'], shot_examples=shot_examples, system_prompt=system_prompt) for _, row in eval_df.iterrows()]

    verify_token_limits(prompts)

    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts = prompts[i:i+batch_size]

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=5,
                temperature=0.0,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        input_lengths = inputs['input_ids'].shape[1]
        generated_tokens = outputs[:, input_lengths:]
        batch_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)

        for pred_text in batch_preds:
            predictions.append(extract_label(pred_text))

    overall_f1 = f1_score(ground_truth, predictions, average='macro')
    print(f"\n--- {batch_name} Classification Report ---")
    print(classification_report(ground_truth, predictions, digits=4, zero_division=0))

    return overall_f1, predictions

In [ ]:
# ==============================================================================
# CELL 5.5: Empirical Prompt A/B Testing (Evidence for Report)
# ==============================================================================
print("Running A/B Prompt Test on Validation Subset...")

# Take a quick 50-sample subset of validation data
val_subset = val_df.head(50)

# Define a short, non-descriptive baseline prompt
SHORT_SYSTEM_PROMPT = "Classify the question into: TTD, TGU, ACM, TRS, WTH, FOD, ENT. Return ONLY the 3-letter abbreviation."

# 1. Test the Short Prompt
f1_short, _ = run_evaluation(
    val_subset,
    shot_examples=[],
    batch_name="Prompt A (Short)",
    batch_size=8,
    system_prompt=SHORT_SYSTEM_PROMPT
)

# 2. Test the Detailed Taxonomy Prompt
f1_detailed, _ = run_evaluation(
    val_subset,
    shot_examples=[],
    batch_name="Prompt B (Detailed Taxonomy)",
    batch_size=8,
    system_prompt=SYSTEM_PROMPT
)

# 3. Print the Empirical Proof
print("\n========================================")
print("   PROMPT ENGINEERING RESULTS (Validation Subset)")
print("========================================")
print(f"Prompt A (Short) Macro F1    : {f1_short:.4f}")
print(f"Prompt B (Detailed) Macro F1 : {f1_detailed:.4f}")
print("========================================")
if f1_detailed >= f1_short:
    print("Conclusion: Detailed taxonomy definitions improve classification performance.")

In [ ]:
# ==============================================================================
# CELL 6: Zero-shot, 1-shot, and 3-shot Prompt Testing (FULL RUN)
# ==============================================================================
import pandas as pd
from IPython.display import display

print("--- Extracting Few-Shot Demonstrations from Training Data ---")

# 1. Select 1-Shot Example (e.g., TTD)
one_shot_raw = train_df[train_df['Coarse_Class'] == 'TTD'].sample(n=1, random_state=42).iloc[0]
one_shot_example = [one_shot_raw.to_dict()]

# 2. Select 3-Shot Examples from distinct, diverse classes
three_shot_raw = []
for cls in ['TTD', 'ACM', 'TRS']:
    sample = train_df[train_df['Coarse_Class'] == cls].sample(n=1, random_state=42).iloc[0]
    three_shot_raw.append(sample)

three_shot_examples = [row.to_dict() for row in three_shot_raw]

# 3. Display the visual proof
print("\n3-Shot Exemplars Selected:")
display(pd.DataFrame(three_shot_raw)[['Question', 'Coarse_Class']])

print("\n--- Beginning Evaluations ---")

# 1. Zero-shot testing
f1_zero, preds_zero = run_evaluation(test_df, shot_examples=[], batch_name="Zero-shot")

# 2. 1-shot testing
f1_one, preds_one = run_evaluation(test_df, shot_examples=one_shot_example, batch_name="1-shot")

# 3. 3-shot testing
f1_three, preds_three = run_evaluation(test_df, shot_examples=three_shot_examples, batch_name="3-shot")

In [ ]:
# ==============================================================================
# CELL 7: Dataset Formatting for Supervised Fine-Tuning (SFT)
# ==============================================================================
from datasets import Dataset

def format_chat_prompt(batch):
    formatted_texts = []
    for question, label in zip(batch['Question'], batch['Coarse_Class']):
        text = (
            f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{SYSTEM_PROMPT}<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\n\nQuestion: {question}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n{label}<|eot_id|>"
        )
        formatted_texts.append(text)
    return {"text": formatted_texts}

train_dataset = Dataset.from_pandas(train_df).map(format_chat_prompt, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(format_chat_prompt, batched=True)

In [ ]:
# ==============================================================================
# CELL 8: Parameter-Efficient Fine-Tuning (PEFT / LoRA) Configuration
# ==============================================================================
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    max_seq_length=max_seq_length,
)

print("LoRA adapters injected successfully.")
model.print_trainable_parameters()

# Defensive Token Length Check for SFT Datasets ---
def check_sft_lengths(dataset, name):
    """Verifies that no training example exceeds the model's context window."""
    lengths = []

    # Iterate row by row to ensure the tokenizer receives standard strings
    for example in dataset:
        tokens = tokenizer(example["text"])
        lengths.append(len(tokens["input_ids"]))

    max_len = max(lengths)
    over_limit = sum(1 for n in lengths if n > max_seq_length)

    print(f"{name} max tokens: {max_len}")
    if over_limit > 0:
        print(f"WARNING: {over_limit} sequences exceed the {max_seq_length} token limit!")
    else:
        print("All sequences are safely within the token limit.")

print("\n--- Verifying SFT Token Lengths ---")
check_sft_lengths(train_dataset, "SFT Train Dataset")
check_sft_lengths(val_dataset, "SFT Validation Dataset")

In [ ]:
# ==============================================================================
# CELL 9: Execute Supervised Fine-Tuning (SFT) - FULL RUN
# ==============================================================================
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

# Define arguments with validation tracking and memory-safe batching
training_args = TrainingArguments(
    output_dir="./outputs_sft",

    # Memory-safe batching (Effective Batch Size = 2 * 4 = 8)
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    # Hyperparameters
    warmup_steps=10,
    max_steps=120,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    report_to="none",

    # Evaluation & Best Checkpoint Selection ---
    eval_strategy="steps",           # Evaluate during training
    eval_steps=20,                   # Check validation loss every 20 steps
    save_steps=20,                   # Save checkpoint every 20 steps
    load_best_model_at_end=True,     # Automatically revert to the best model!
    metric_for_best_model="eval_loss",
    greater_is_better=False,         # Lower loss is better
    logging_steps=10,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print("Starting Supervised Fine-Tuning with Validation Tracking...")
trainer_stats = trainer.train()
print("Supervised Fine-Tuning completed! Best model automatically loaded.")

In [ ]:
# ==============================================================================
# CELL 10: Final Evaluation & Results Comparison Table - FULL RUN
# ==============================================================================
# Switch model back to inference mode
FastLanguageModel.for_inference(model)

# Evaluate fine-tuned model on full test set
f1_sft, preds_sft = run_evaluation(test_df, shot_examples=[], batch_name="SFT Model")

# Calculate overall class-wise F1 across experiments
ground_truth = test_df['Coarse_Class'].tolist()

report_sft = classification_report(ground_truth, preds_sft, output_dict=True, zero_division=0)
report_zero = classification_report(ground_truth, preds_zero, output_dict=True, zero_division=0)
report_one = classification_report(ground_truth, preds_one, output_dict=True, zero_division=0)
report_three = classification_report(ground_truth, preds_three, output_dict=True, zero_division=0)

# Build formatted evaluation benchmark table
results_data = {
    "Class": VALID_CLASSES + ["Macro Avg F1", "Weighted Avg F1"],
    "Zero-shot": [round(report_zero.get(c, {}).get('f1-score', 0.0), 4) for c in VALID_CLASSES] + [round(report_zero['macro avg']['f1-score'], 4), round(report_zero['weighted avg']['f1-score'], 4)],
    "1-shot": [round(report_one.get(c, {}).get('f1-score', 0.0), 4) for c in VALID_CLASSES] + [round(report_one['macro avg']['f1-score'], 4), round(report_one['weighted avg']['f1-score'], 4)],
    "3-shot": [round(report_three.get(c, {}).get('f1-score', 0.0), 4) for c in VALID_CLASSES] + [round(report_three['macro avg']['f1-score'], 4), round(report_three['weighted avg']['f1-score'], 4)],
    "SFT": [round(report_sft.get(c, {}).get('f1-score', 0.0), 4) for c in VALID_CLASSES] + [round(report_sft['macro avg']['f1-score'], 4), round(report_sft['weighted avg']['f1-score'], 4)],
}

results_df = pd.DataFrame(results_data)
print("\n================== FINAL EXPERIMENT RESULTS TABLE ==================")
print(results_df.to_markdown(index=False))

In [ ]:
# ==============================================================================
# CELL 11: SFT CONFUSION MATRIX VISUALIZATION
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

print("Generating Confusion Matrix for the Final SFT Model...")

# Calculate the matrix using your exact variables
cm = confusion_matrix(
    test_df['Coarse_Class'],
    preds_sft,
    labels=VALID_CLASSES
)

# Set up the plot aesthetics for a professional report
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,       # Show the actual numbers
    fmt="d",          # Format as integers
    cmap="Blues",     # A clean, academic blue color scheme
    xticklabels=VALID_CLASSES,
    yticklabels=VALID_CLASSES
)

plt.xlabel("Predicted Category", fontweight='bold')
plt.ylabel("Actual Category", fontweight='bold')
plt.title("Supervised Fine-Tuned (SFT) Model - Confusion Matrix", fontweight='bold', pad=15)
plt.show()

In [ ]:
# ==============================================================================
# CELL 12: QUALITATIVE ANALYSIS (Errors & Successes)
# ==============================================================================
from IPython.display import display

def display_errors(df, predictions, model_name, num_errors=10):
    """Finds and displays misclassifications."""
    analysis_df = df.copy()
    analysis_df['Prediction'] = predictions
    errors = analysis_df[analysis_df['Coarse_Class'] != analysis_df['Prediction']]

    print(f"\n================ {model_name} ERRORS ({len(errors)} total mistakes) ================")
    display(errors[['Question', 'Coarse_Class', 'Prediction']].head(num_errors))

def display_successes(df, predictions, model_name, num_successes=5):
    """Finds and displays successful classifications."""
    analysis_df = df.copy()
    analysis_df['Prediction'] = predictions
    successes = analysis_df[analysis_df['Coarse_Class'] == analysis_df['Prediction']]

    print(f"\n================ {model_name} SUCCESSES ================")
    display(successes[['Question', 'Coarse_Class', 'Prediction']].head(num_successes))


# 1. Look at the errors across all models to see how they improved
display_errors(test_df, preds_zero, "ZERO-SHOT")
display_errors(test_df, preds_one, "1-SHOT")
display_errors(test_df, preds_three, "3-SHOT")
display_errors(test_df, preds_sft, "SUPERVISED FINE-TUNED (SFT)")

# 2. Show correct examples for the final SFT model to highlight its strengths!
display_successes(test_df, preds_sft, "SUPERVISED FINE-TUNED (SFT)")

In [ ]:
# ==============================================================================
# CELL 13: SAVE THE FINE-TUNED MODEL (LoRA Adapters)
# ==============================================================================

OUTPUT_DIR = "./final_travel_model_lora"

print(f"Saving LoRA adapters and tokenizer to {OUTPUT_DIR}...")

# This efficiently saves ONLY the trained LoRA weights, not the whole 8B model!
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved successfully!")

In [ ]:
# ==============================================================================
# CELL 14: SAVE EXPERIMENT RESULTS TO CSV
# ==============================================================================
print("Saving all experiment results to CSV...")

# 1. Save master results table (The one from Cell 10)
results_df.to_csv("final_experiment_metrics.csv", index=False)

# 2. Build a single, comprehensive predictions DataFrame
predictions_export = test_df.copy()
predictions_export['Zero_Shot_Pred'] = preds_zero
predictions_export['One_Shot_Pred'] = preds_one
predictions_export['Three_Shot_Pred'] = preds_three
predictions_export['SFT_Pred'] = preds_sft

# Save all predictions side-by-side
predictions_export.to_csv("all_model_predictions.csv", index=False)

print("Successfully saved:")
print("- final_experiment_metrics.csv (Contains your exact report tables)")
print("- all_model_predictions.csv (Contains every question and model guess side-by-side)")